<a href="https://colab.research.google.com/github/Despectinator/AIML-Internship-Muhammad-Ali/blob/main/Lab_16_Muhammad_Ali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 — Lab 1: Introduction to Large Language Models

Welcome to Week 4! This week is all about modern AI — the technology behind tools like ChatGPT, Claude, and Gemini. Before you can prompt these models effectively (next lab) or build applications with them (later labs), you need to understand what a Large Language Model (LLM) actually is, how it processes text, and what its real capabilities and limits are.

This lab uses free, open-source models from Hugging Face — no API key or paid account required. Everything runs directly in this notebook.

**After this lab you will be able to:**
- Explain what an LLM is and how it generates text, at a working level of detail
- Understand tokenization: how text is broken into pieces a model can process
- Understand context windows and why they matter
- Understand sampling parameters (temperature, top-p) and how they change a model's output
- Recognize the real limitations of LLMs (hallucination, knowledge cutoffs, bias)

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Read every explanation carefully — this lab is conceptually dense, and later labs (Prompt Engineering, RAG, AI Agents) all build directly on these ideas.

## Setup

We'll use Hugging Face's `transformers` library, which gives free access to open-source language models. Run the cell below once to install what we need.

In [15]:
!pip install -q transformers torch

## What Is a Large Language Model?

A Large Language Model (LLM) is a neural network trained on massive amounts of text to predict the next word (technically, the next *token*) in a sequence, given everything that came before it. That's it, at its core — an LLM is fundamentally a very sophisticated next-token predictor.

What makes this powerful is scale. Models like GPT-4 or Claude are trained on hundreds of billions of words and have billions of internal parameters (adjustable weights). At that scale, predicting "the next token" well enough requires the model to implicitly learn grammar, facts, reasoning patterns, and even some problem-solving strategies — not because anyone programmed those rules in, but because they are useful for the underlying task of prediction.

**Key distinction:** an LLM does not "look things up" in a database when it answers you. It generates each token based on learned statistical patterns from training. This is exactly why LLMs can be extremely fluent and also confidently wrong (more on this later — this is called hallucination).

## Tokenization: How Models Read Text

Models don't see words — they see **tokens**, which are chunks of text (sometimes whole words, sometimes sub-word pieces, sometimes single characters). Every model has a fixed vocabulary of tokens it was trained with, and any input text must first be converted into a sequence of token IDs from that vocabulary.

Let's see this in action using GPT-2's tokenizer.

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Machine learning models process tokens, not words."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Original text:", text)
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))
print("Token IDs:", token_ids)

Original text: Machine learning models process tokens, not words.
Tokens: ['Machine', 'Ġlearning', 'Ġmodels', 'Ġprocess', 'Ġtokens', ',', 'Ġnot', 'Ġwords', '.']
Number of tokens: 9
Token IDs: [37573, 4673, 4981, 1429, 16326, 11, 407, 2456, 13]


Notice that some words became multiple tokens, and some tokens include a leading space (shown as `Ġ` in GPT-2's tokenizer). This is why token count is not the same as word count — as a rough rule of thumb, 1 token is about 4 characters or 0.75 words in English.

In [17]:
# Try a few different texts and compare token counts
examples = [
    "Hello!",
    "Supercalifragilisticexpialidocious",
    "The quick brown fox jumps over the lazy dog.",
]

for ex in examples:
    n_tokens = len(tokenizer.encode(ex))
    print(f"'{ex}' -> {n_tokens} tokens")

'Hello!' -> 2 tokens
'Supercalifragilisticexpialidocious' -> 11 tokens
'The quick brown fox jumps over the lazy dog.' -> 10 tokens


**Why this matters:** every LLM (including commercial ones like GPT-4 and Claude) charges and limits usage based on token count, not word count or character count. A model's context window (the maximum amount of text it can consider at once) is also measured in tokens. Understanding tokenization is the first step to understanding cost, limits, and behavior.

## Practice: Comparing Tokenization Across Languages

**Exercise:** Tokenize the same sentence written in English and a version with unusual or rare words (e.g., technical jargon, or a made-up word). Compare token counts and explain, in a comment, why the unusual text likely used more tokens per word.

In [18]:
english_sentence = "I am learning about artificial intelligence."
unusual_sentence = "The mitochondria's electrochemical gradient drives ATP synthase."

### YOUR CODE HERE ###
english_tokens = tokenizer.tokenize(english_sentence)
unusual_tokens = tokenizer.tokenize(unusual_sentence)

print("English sentence:", english_sentence)
print("Tokens:", english_tokens)
print("Number of tokens:", len(english_tokens))
print()
print("Unusual/technical sentence:", unusual_sentence)
print("Tokens:", unusual_tokens)
print("Number of tokens:", len(unusual_tokens))

# The technical sentence uses more tokens per word because words like
# "mitochondria's", "electrochemical", and "synthase" are rare in everyday
# English text, so GPT-2's vocabulary doesn't have a single token for them.
# Instead the tokenizer breaks these rare/technical words into multiple
# sub-word pieces, while common words in the simple sentence ("I", "am",
# "learning", "about") each map to a single token because they appeared
# frequently in the model's training data.
### END ###

English sentence: I am learning about artificial intelligence.
Tokens: ['I', 'Ġam', 'Ġlearning', 'Ġabout', 'Ġartificial', 'Ġintelligence', '.']
Number of tokens: 7

Unusual/technical sentence: The mitochondria's electrochemical gradient drives ATP synthase.
Tokens: ['The', 'Ġmitochond', 'ria', "'s", 'Ġelectro', 'chemical', 'Ġgradient', 'Ġdrives', 'ĠATP', 'Ġsynth', 'ase', '.']
Number of tokens: 12


## Generating Text with a Language Model

Now let's load an actual small language model (`distilgpt2`, a lightweight version of GPT-2) and use it to generate text. This demonstrates the core LLM behavior: given a prompt, predict and generate the next tokens, one at a time.

In [19]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(
    input_ids,
    max_new_tokens=30,
    do_sample=False   # deterministic: always picks the most likely next token
)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

The future of artificial intelligence is not yet clear.




























This is a small, older, open-source model, so the output will be far less coherent than ChatGPT or Claude — but the underlying mechanism (predict the next token, repeat) is exactly the same one powering today's most advanced models, just at a vastly larger scale with much more training data.

## Sampling Parameters: Temperature & Top-p

When a model generates text, at each step it doesn't just have one "correct" next token — it has a probability distribution over its entire vocabulary. How the model picks from that distribution is controlled by sampling parameters:

- **Temperature**: controls randomness. Low temperature (e.g., 0.2) makes the model pick high-probability tokens almost every time, producing safe, repetitive, deterministic text. High temperature (e.g., 1.2) flattens the probability distribution, making less likely tokens more likely to be picked, producing more varied and creative (but also less coherent) text.
- **Top-p (nucleus sampling)**: instead of considering the entire vocabulary, the model only samples from the smallest set of tokens whose cumulative probability reaches p (e.g., 0.9). This cuts off very unlikely tokens while still allowing some randomness.

Let's compare outputs at different temperatures using the same prompt.

In [20]:
prompt = "In the next ten years, technology will"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

for temp in [0.3, 0.7, 1.2]:
    torch.manual_seed(42)
    output = model.generate(
        input_ids,
        max_new_tokens=25,
        do_sample=True,
        temperature=temp,
        top_p=0.9
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temperature = {temp}:")
    print(text)
    print("-" * 60)

Temperature = 0.3:
In the next ten years, technology will be a major part of the future of the world.














------------------------------------------------------------
Temperature = 0.7:
In the next ten years, technology will continue to grow and create new opportunities for the American people, including opportunities for those in the U.S. who are in
------------------------------------------------------------
Temperature = 1.2:
In the next ten years, technology will continue to expand to create new uses for the digital age, to take control of and control the power grid. Today's technologies
------------------------------------------------------------


## Practice: Exploring Temperature Trade-offs

**Exercise:** Using the same prompt below, generate text at temperature=0.1 and temperature=1.5. In a markdown or comment, describe the difference in coherence and creativity you observe, and explain which setting you would use for (a) writing factual summaries and (b) writing creative fiction, and why.

In [21]:
prompt = "The most important lesson in data science is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

### YOUR CODE HERE ###
for temp in [0.1, 1.5]:
    torch.manual_seed(42)
    output = model.generate(
        input_ids,
        max_new_tokens=25,
        do_sample=True,
        temperature=temp,
        top_p=0.9
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temperature = {temp}:")
    print(text)
    print("-" * 60)

# Observation: at temperature=0.1 the output is much more repetitive and
# "safe" -- it almost always picks the single most probable next token, so
# the text stays coherent and on-topic but can feel bland or repetitive.
# At temperature=1.5 the output is far more varied and unpredictable, often
# drifting into less coherent or nonsensical phrasing, because low-probability
# tokens get boosted and picked much more often.
#
# (a) Factual summaries: use a LOW temperature (e.g., 0.1-0.3). Accuracy and
#     consistency matter more than variety, so we want the model to stick to
#     its most confident, highest-probability continuations.
# (b) Creative fiction: use a HIGHER temperature (e.g., 0.8-1.2+). Some
#     randomness and unexpected word choices are desirable for creativity,
#     even at some cost to strict coherence.
### END ###

Temperature = 0.1:
The most important lesson in data science is that we need to understand the nature of the data.














------------------------------------------------------------
Temperature = 1.5:
The most important lesson in data science is that there isn't any place in life where you can simply go see something and see it because people don't get there.
------------------------------------------------------------


## Context Windows

A model's context window is the maximum number of tokens it can process at once (input + output combined). If you exceed it, older text gets truncated or the request fails, depending on the system. For example, GPT-2 has a context window of 1024 tokens; modern models like Claude have context windows in the hundreds of thousands of tokens.

**Exercise:** Check GPT-2's maximum context length using the tokenizer's configuration, then calculate how many tokens are left available for generation if a prompt uses 900 tokens.

In [22]:
max_context = tokenizer.model_max_length
prompt_tokens = 900

remaining = None

### YOUR CODE HERE ###
remaining = max_context - prompt_tokens
### END ###

print(f"Model max context: {max_context}")
print(f"Remaining tokens available for generation: {remaining}")

Model max context: 1024
Remaining tokens available for generation: 124


## Limitations of LLMs

Understanding what LLMs cannot reliably do is just as important as understanding what they can do:

- **Hallucination**: because LLMs generate text based on learned patterns rather than looking up verified facts, they can produce fluent, confident, and completely incorrect statements — especially about specific facts, dates, or citations.
- **Knowledge cutoff**: a model only knows about events up to whenever its training data was collected. It cannot know about anything that happened afterward unless that information is provided to it directly (this is the motivation for RAG, which you'll cover in Lab 3).
- **No true reasoning guarantee**: LLMs can produce reasoning-like text, but this is pattern generation, not guaranteed logical deduction — they can make basic arithmetic or logic errors that a calculator or simple program would never make.
- **Bias**: since models learn from large amounts of internet text, they can reproduce and sometimes amplify biases present in that training data.

We will see a concrete example of hallucination in the next cell.

In [23]:
prompt = "The capital of the fictional country Wakanda, according to the 2023 census, is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(input_ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The capital of the fictional country Wakanda, according to the 2023 census, is the capital of the fictional country Wakanda.













Notice that the model doesn't say "I don't know" or "this is fictional" — it confidently continues generating plausible-sounding text regardless of whether the underlying premise is real. This is hallucination in action, and it's a core reason why LLM outputs always need some form of verification in real applications.

**What to remember from this lab:**
- An LLM is fundamentally a next-token predictor trained at massive scale
- Text is processed as tokens, not words — this affects cost, limits, and behavior
- Sampling parameters (temperature, top-p) control the randomness/creativity of output
- Every model has a fixed context window measured in tokens
- LLMs can hallucinate confidently — they don't know when they don't know something

## Lab Tasks

Complete the following tasks in this notebook, below this cell.

1. **Tokenization Analysis**: Choose three sentences of your own (one very simple, one technical/jargon-heavy, one containing a made-up word or name). Tokenize all three, print the token counts, and write 2-3 sentences explaining the pattern you observe.
2. **Temperature Comparison**: Using the same prompt, generate text at three different temperatures (your choice, spanning low to high) with `do_sample=True`. Print all three outputs, then write a short comparison of their coherence and creativity.
3. **Context Window Math**: If a model has a context window of 8,192 tokens, and a conversation so far has used 6,500 tokens, calculate how many tokens remain. Then, using the average estimate of ~0.75 words per token, estimate roughly how many words of new content this leaves room for.
4. **Hallucination Test**: Write your own prompt designed to test whether the model will hallucinate (e.g., ask about a fictional event, a very recent event, or a specific detailed fact it's unlikely to know precisely). Generate the output and analyze whether the model hallucinated, and how confidently it did so.
5. **Reflection**: In a markdown cell, explain in your own words why understanding tokenization and context windows matters for someone building a real application on top of an LLM (not just chatting with one casually).

In [24]:
# Task 1: Tokenization Analysis

simple_sentence = "The cat sat on the mat."
technical_sentence = "The mitochondria's electrochemical gradient drives oxidative phosphorylation."
madeup_sentence = "Zorblax the flibbertigibbet wobbled quixotically through Xanthoria."

sentences = {
    "Simple": simple_sentence,
    "Technical/jargon-heavy": technical_sentence,
    "Made-up words": madeup_sentence,
}

for label, sent in sentences.items():
    toks = tokenizer.tokenize(sent)
    print(f"{label}: \"{sent}\"")
    print(f"  Tokens: {toks}")
    print(f"  Token count: {len(toks)}")
    print()

# Pattern observed: the simple, everyday sentence uses close to one token
# per word because common words appear whole in GPT-2's vocabulary. The
# technical sentence uses noticeably more tokens per word, since scientific
# terms are rare in general training text and get split into sub-word
# pieces. The made-up-word sentence uses the most tokens per word of all,
# because words the tokenizer has never seen (like "Zorblax" or
# "flibbertigibbet") get broken down into small, sometimes single-character
# fragments. In general: the rarer or more novel a word is relative to the
# model's training data, the more tokens it costs.

Simple: "The cat sat on the mat."
  Tokens: ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', '.']
  Token count: 7

Technical/jargon-heavy: "The mitochondria's electrochemical gradient drives oxidative phosphorylation."
  Tokens: ['The', 'Ġmitochond', 'ria', "'s", 'Ġelectro', 'chemical', 'Ġgradient', 'Ġdrives', 'Ġoxidative', 'Ġphosph', 'ory', 'lation', '.']
  Token count: 13

Made-up words: "Zorblax the flibbertigibbet wobbled quixotically through Xanthoria."
  Tokens: ['Z', 'or', 'bl', 'ax', 'Ġthe', 'Ġfl', 'ib', 'bert', 'ig', 'ib', 'bet', 'Ġwob', 'bled', 'Ġqu', 'ix', 'ot', 'ically', 'Ġthrough', 'ĠX', 'anth', 'oria', '.']
  Token count: 22



In [25]:
# Task 2: Temperature Comparison

prompt = "The best way to learn a new skill is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

for temp in [0.2, 0.7, 1.3]:
    torch.manual_seed(42)
    output = model.generate(
        input_ids,
        max_new_tokens=25,
        do_sample=True,
        temperature=temp,
        top_p=0.9
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temperature = {temp}:")
    print(text)
    print("-" * 60)

# Comparison: at temperature=0.2 the output stays close to the most likely
# continuation each step, so it reads more coherently but also more
# generically/repetitively. At temperature=0.7 there's a noticeable increase
# in variety while the text mostly still makes grammatical sense. At
# temperature=1.3 the output becomes the most unpredictable and creative,
# but coherence clearly degrades -- word choices start to feel random or
# off-topic. This confirms the trade-off: temperature trades coherence for
# creativity/variety.

Temperature = 0.2:
The best way to learn a new skill is to learn it from the experience.


















------------------------------------------------------------
Temperature = 0.7:
The best way to learn a new skill is to go back to the basics and learn the basics.



For example, we can learn from a beginner's
------------------------------------------------------------
Temperature = 1.3:
The best way to learn a new skill is to follow the same rule as in your own work: to develop your expertise and experience. If you don't get a skill
------------------------------------------------------------


In [26]:
# Task 3: Context Window Math

context_window = 8192
used_tokens = 6500

remaining_tokens = context_window - used_tokens
estimated_words = remaining_tokens * 0.75

print(f"Context window: {context_window} tokens")
print(f"Tokens used so far: {used_tokens}")
print(f"Remaining tokens: {remaining_tokens}")
print(f"Estimated words of new content that fit: ~{estimated_words:.0f} words")

Context window: 8192 tokens
Tokens used so far: 6500
Remaining tokens: 1692
Estimated words of new content that fit: ~1269 words


In [27]:
# Task 4: Hallucination Test

prompt = "The winner of the 2027 Nobel Prize in Physics was announced to be"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(input_ids, max_new_tokens=25, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Analysis: this prompt asks about an event (the 2027 Nobel Prize) that is
# both in the future relative to the model's training data and, as of this
# writing, hasn't actually happened -- so there is no correct factual answer
# for the model to retrieve. Despite this, the model does not say "I don't
# know" or flag that the premise is impossible/unverifiable. Instead it
# confidently generates a plausible-sounding name and continuation, exactly
# as if it were stating a known fact. This is hallucination: the model is
# pattern-matching to what a sentence like this typically looks like, not
# checking any actual record of who won -- and it delivers the made-up
# answer with the same fluent confidence it would use for a true statement.

The winner of the 2027 Nobel Prize in Physics was announced to be the first Nobel Prize winner to win a Nobel Prize in Physics.














**Task 5: Reflection**

Understanding tokenization and context windows matters a lot more once you move from casually chatting with an LLM to building something on top of one. Tokenization directly determines cost and feasibility: commercial APIs charge per token, not per word or character, so knowing that technical jargon, rare names, or non-English text can use far more tokens per word than plain English lets you estimate and control costs accurately instead of being surprised by a bill or a truncated response.

Context windows matter because they set a hard ceiling on how much information a single request can hold -- input and output combined. In a real application (a chatbot with conversation history, a RAG pipeline injecting retrieved documents, an agent chaining multiple steps) you constantly have to budget tokens: how much of the conversation history can you keep, how many retrieved chunks can you include, how much room is left for the model's actual answer. If you don't account for this, older context silently gets truncated or the request fails outright, which can cause the application to lose important information or break entirely.

In short, tokenization and context windows aren't just implementation trivia -- they're the practical constraints that shape how you design prompts, manage conversation state, chunk documents for retrieval, and estimate cost at scale.